<a href="https://colab.research.google.com/github/WagAtoguiaJr/Trabalho_M3_Banco_Dados_1_Projeto_Uber/blob/main/CRUD_projeto_UBER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CRUD Motoristas e Veículos
Este notebook cria o esquema SQLite, popula dados de exemplo e fornece funções para gerenciar **motoristas** e **veículos**.


In [1]:
# Código: inicialização do banco e esquema
import sqlite3
from sqlite3 import Connection
import datetime
from typing import Optional, List, Dict

DB_FILE = "projeto_uber.db"

SCHEMA_SQL = """
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS cores_veiculos (
  id_cor INTEGER PRIMARY KEY AUTOINCREMENT,
  nome_cor TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS marcas_veiculos (
  id_marca INTEGER PRIMARY KEY AUTOINCREMENT,
  nome_marca TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS modelos_veiculos (
  id_modelo INTEGER PRIMARY KEY AUTOINCREMENT,
  nome_modelo TEXT NOT NULL,
  id_marca INTEGER NOT NULL,
  lugares INTEGER,
  ano_fabricacao INTEGER,
  FOREIGN KEY (id_marca) REFERENCES marcas_veiculos(id_marca) ON UPDATE CASCADE ON DELETE RESTRICT
);

CREATE TABLE IF NOT EXISTS motoristas (
  id_motorista INTEGER PRIMARY KEY AUTOINCREMENT,
  nome_completo TEXT NOT NULL,
  cpf TEXT UNIQUE,
  cnh TEXT UNIQUE,
  data_nasc TEXT,
  email TEXT,
  data_entrada TEXT DEFAULT (datetime('now')),
  ativo INTEGER DEFAULT 1
);

CREATE TABLE IF NOT EXISTS telefones_motoristas (
  id_telefone_motorista INTEGER PRIMARY KEY AUTOINCREMENT,
  id_motorista INTEGER NOT NULL,
  numero TEXT NOT NULL,
  tipo_telefone TEXT DEFAULT 'celular',
  FOREIGN KEY (id_motorista) REFERENCES motoristas(id_motorista) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS enderecos_motoristas (
  id_endereco_motorista INTEGER PRIMARY KEY AUTOINCREMENT,
  id_motorista INTEGER NOT NULL,
  rua TEXT,
  numero TEXT,
  complemento TEXT,
  bairro TEXT,
  cidade TEXT,
  estado TEXT,
  cep TEXT,
  pais TEXT DEFAULT 'Brasil',
  FOREIGN KEY (id_motorista) REFERENCES motoristas(id_motorista) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS veiculos (
  id_veiculo INTEGER PRIMARY KEY AUTOINCREMENT,
  id_motorista INTEGER NOT NULL,
  placa TEXT NOT NULL UNIQUE,
  ativo INTEGER DEFAULT 1,
  id_modelo INTEGER,
  id_cor INTEGER,
  FOREIGN KEY (id_motorista) REFERENCES motoristas(id_motorista) ON UPDATE CASCADE ON DELETE RESTRICT,
  FOREIGN KEY (id_modelo) REFERENCES modelos_veiculos(id_modelo) ON UPDATE CASCADE ON DELETE SET NULL,
  FOREIGN KEY (id_cor) REFERENCES cores_veiculos(id_cor) ON UPDATE CASCADE ON DELETE SET NULL
);
"""

def get_conn() -> Connection:
    conn = sqlite3.connect(DB_FILE)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn

def init_db():
    conn = get_conn()
    conn.executescript(SCHEMA_SQL)
    conn.commit()
    conn.close()

# Inicializa o banco
init_db()
print("Banco inicializado:", DB_FILE)


Banco inicializado: projeto_uber.db


## Popular dados de exemplo
Insere 10 registros coerentes em cada tabela auxiliar, 10 motoristas e 10 veículos.


In [2]:
# Código: seed de dados de exemplo
def seed_sample_data():
    conn = get_conn()
    cur = conn.cursor()
    # cores
    cores = ['Preto','Branco','Prata','Azul','Vermelho','Cinza','Verde','Amarelo','Marrom','Roxo']
    for c in cores:
        try:
            cur.execute("INSERT INTO cores_veiculos (nome_cor) VALUES (?)", (c,))
        except sqlite3.IntegrityError:
            pass
    # marcas
    marcas = ['Toyota','Chevrolet','Volkswagen','Ford','Hyundai','Nissan','Renault','Honda','Kia','Fiat']
    for m in marcas:
        try:
            cur.execute("INSERT INTO marcas_veiculos (nome_marca) VALUES (?)", (m,))
        except sqlite3.IntegrityError:
            pass
    # modelos
    modelos = [
        ('Corolla', 1, 5, 2018),
        ('Onix', 2, 5, 2020),
        ('Gol', 3, 5, 2017),
        ('Ka', 4, 5, 2019),
        ('HB20', 5, 5, 2021),
        ('Sentra', 6, 5, 2016),
        ('Sandero', 7, 5, 2015),
        ('Civic', 8, 5, 2018),
        ('Sportage', 9, 5, 2020),
        ('Cronos', 10, 5, 2019)
    ]
    for nome, id_marca, lugares, ano in modelos:
        try:
            cur.execute("""
                INSERT INTO modelos_veiculos (nome_modelo, id_marca, lugares, ano_fabricacao)
                VALUES (?, ?, ?, ?)
            """, (nome, id_marca, lugares, ano))
        except sqlite3.IntegrityError:
            pass

    # motoristas
    motoristas = [
        ('João Souza','111.222.333-44','CNH12345','1982-02-20','joao.souza@example.com',1),
        ('Marcos Lima','555.666.777-88','CNH67890','1979-07-15','marcos.lima@example.com',1),
        ('Paula Rocha','999.888.777-66','CNH54321','1990-12-01','paula.rocha@example.com',1),
        ('Eduardo Nunes','121.212.121-12','CNH11111','1987-04-04','eduardo.nunes@example.com',1),
        ('Sofia Almeida','131.313.131-13','CNH22222','1992-10-10','sofia.almeida@example.com',1),
        ('Bruno Castro','141.414.141-14','CNH33333','1985-06-06','bruno.castro@example.com',1),
        ('Camila Pinto','151.515.151-15','CNH44444','1994-09-09','camila.pinto@example.com',1),
        ('Diego Martins','161.616.161-16','CNH55555','1980-01-01','diego.martins@example.com',1),
        ('Larissa Freitas','171.717.171-17','CNH66666','1991-11-11','larissa.freitas@example.com',1),
        ('Gustavo Rocha','181.818.181-18','CNH77777','1983-08-08','gustavo.rocha@example.com',1)
    ]
    for m in motoristas:
        try:
            cur.execute("""
                INSERT INTO motoristas (nome_completo, cpf, cnh, data_nasc, email, ativo)
                VALUES (?, ?, ?, ?, ?, ?)
            """, m)
        except sqlite3.IntegrityError:
            pass

    # telefones e endereços motoristas
    for i in range(1, 11):
        try:
            cur.execute("INSERT INTO telefones_motoristas (id_motorista, numero) VALUES (?, ?)",
                        (i, f'+55 47 98888-{1000+i}'))
        except sqlite3.IntegrityError:
            pass
        try:
            cur.execute("""
                INSERT INTO enderecos_motoristas (id_motorista, rua, numero, bairro, cidade, estado, cep)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (i, f'Rua Motorista {i}', str(10+i), 'Centro', 'Itajaí', 'SC', f'8830{i}-000'))
        except sqlite3.IntegrityError:
            pass

    # veiculos
    placas = ['ABC1D23','XYZ9K87','MNO4P56','QWE7R45','RTY6U32','FGH5J21','VBN8M90','PLK3O11','IJU2Y33','ASD4F55']
    for i in range(1, 11):
        try:
            cur.execute("""
                INSERT INTO veiculos (id_motorista, placa, ativo, id_modelo, id_cor)
                VALUES (?, ?, ?, ?, ?)
            """, (i, placas[i-1], 1, i, i))
        except sqlite3.IntegrityError:
            pass

    conn.commit()
    conn.close()
    print("Dados de exemplo inseridos.")

# Executar seed
seed_sample_data()


Dados de exemplo inseridos.


## Funções CRUD para Motoristas
As funções abaixo permitem criar, listar, visualizar, atualizar e deletar motoristas.


In [3]:
# Código: CRUD motoristas
def create_motorista(nome: str, cpf: Optional[str]=None, cnh: Optional[str]=None,
                     data_nasc: Optional[str]=None, email: Optional[str]=None, ativo: bool=True) -> int:
    conn = get_conn()
    cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO motoristas (nome_completo, cpf, cnh, data_nasc, email, ativo)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (nome, cpf, cnh, data_nasc, email, 1 if ativo else 0))
        conn.commit()
        return cur.lastrowid
    finally:
        conn.close()

def list_motoristas() -> List[Dict]:
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SELECT * FROM motoristas ORDER BY id_motorista")
    rows = [dict(r) for r in cur.fetchall()]
    conn.close()
    return rows

def get_motorista(id_motorista: int) -> Optional[Dict]:
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SELECT * FROM motoristas WHERE id_motorista = ?", (id_motorista,))
    row = cur.fetchone()
    conn.close()
    return dict(row) if row else None

def update_motorista(id_motorista: int, **fields) -> bool:
    allowed = ['nome_completo','cpf','cnh','data_nasc','email','ativo']
    set_parts = []
    params = []
    for k, v in fields.items():
        if k in allowed:
            set_parts.append(f"{k} = ?")
            params.append(v)
    if not set_parts:
        return False
    params.append(id_motorista)
    sql = f"UPDATE motoristas SET {', '.join(set_parts)} WHERE id_motorista = ?"
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(sql, params)
    conn.commit()
    changed = cur.rowcount > 0
    conn.close()
    return changed

def delete_motorista(id_motorista: int) -> bool:
    conn = get_conn()
    cur = conn.cursor()
    # verifica veículos vinculados
    cur.execute("SELECT COUNT(*) as cnt FROM veiculos WHERE id_motorista = ?", (id_motorista,))
    cnt = cur.fetchone()['cnt']
    if cnt > 0:
        conn.close()
        raise Exception(f"Existem {cnt} veículo(s) vinculados. Remova ou reatribua-os antes de deletar.")
    cur.execute("DELETE FROM motoristas WHERE id_motorista = ?", (id_motorista,))
    conn.commit()
    deleted = cur.rowcount > 0
    conn.close()
    return deleted


## Funções CRUD para Veículos
As funções abaixo permitem criar, listar, visualizar, atualizar e deletar veículos.


In [4]:
# Código: CRUD veículos
def create_veiculo(id_motorista: int, placa: str, id_modelo: Optional[int]=None,
                   id_cor: Optional[int]=None, ativo: bool=True) -> int:
    conn = get_conn()
    cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO veiculos (id_motorista, placa, ativo, id_modelo, id_cor)
            VALUES (?, ?, ?, ?, ?)
        """, (id_motorista, placa, 1 if ativo else 0, id_modelo, id_cor))
        conn.commit()
        return cur.lastrowid
    finally:
        conn.close()

def list_veiculos() -> List[Dict]:
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("""
        SELECT v.id_veiculo, v.placa, v.ativo, v.id_motorista, m.nome_completo as motorista,
               v.id_modelo, mo.nome_modelo as modelo, v.id_cor, c.nome_cor as cor
        FROM veiculos v
        LEFT JOIN motoristas m ON v.id_motorista = m.id_motorista
        LEFT JOIN modelos_veiculos mo ON v.id_modelo = mo.id_modelo
        LEFT JOIN cores_veiculos c ON v.id_cor = c.id_cor
        ORDER BY v.id_veiculo
    """)
    rows = [dict(r) for r in cur.fetchall()]
    conn.close()
    return rows

def get_veiculo(id_veiculo: int) -> Optional[Dict]:
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SELECT * FROM veiculos WHERE id_veiculo = ?", (id_veiculo,))
    row = cur.fetchone()
    conn.close()
    return dict(row) if row else None

def update_veiculo(id_veiculo: int, **fields) -> bool:
    allowed = ['placa','id_motorista','id_modelo','id_cor','ativo']
    set_parts = []
    params = []
    for k, v in fields.items():
        if k in allowed:
            set_parts.append(f"{k} = ?")
            params.append(v)
    if not set_parts:
        return False
    params.append(id_veiculo)
    sql = f"UPDATE veiculos SET {', '.join(set_parts)} WHERE id_veiculo = ?"
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(sql, params)
    conn.commit()
    changed = cur.rowcount > 0
    conn.close()
    return changed

def delete_veiculo(id_veiculo: int) -> bool:
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("DELETE FROM veiculos WHERE id_veiculo = ?", (id_veiculo,))
    conn.commit()
    deleted = cur.rowcount > 0
    conn.close()
    return deleted


## Funções CRUD para Marcas, Modelos e Cores
As funções abaixo permitem criar, listar, visualizar, atualizar e deletar Marcas, Modelos e Cores.



In [5]:
def create_marca(nome_marca: str) -> int:
    conn = get_conn()
    cur = conn.cursor()
    try:
        cur.execute("INSERT INTO marcas_veiculos (nome_marca) VALUES (?)", (nome_marca,))
        conn.commit()
        return cur.lastrowid
    except sqlite3.IntegrityError:
        # já existe: retornar id existente
        cur.execute("SELECT id_marca FROM marcas_veiculos WHERE nome_marca = ?", (nome_marca,))
        row = cur.fetchone()
        return row['id_marca'] if row else None
    finally:
        conn.close()

def update_marca(id_marca: int, novo_nome: str) -> bool:
    if not novo_nome or not str(id_marca).isdigit():
        return False
    conn = get_conn()
    cur = conn.cursor()
    try:
        cur.execute("UPDATE marcas_veiculos SET nome_marca = ? WHERE id_marca = ?", (novo_nome.strip(), id_marca))
        conn.commit()
        return cur.rowcount > 0
    finally:
        conn.close()

def delete_marca(id_marca: int) -> bool:
    conn = get_conn()
    cur = conn.cursor()
    try:
        # Verificar existência de modelos vinculados (ON DELETE RESTRICT no esquema)
        cur.execute("SELECT COUNT(*) AS cnt FROM modelos_veiculos WHERE id_marca = ?", (id_marca,))
        cnt = cur.fetchone()['cnt']
        if cnt > 0:
            raise Exception(f"Não é possível deletar: existem {cnt} modelo(s) vinculados a esta marca.")
        cur.execute("DELETE FROM marcas_veiculos WHERE id_marca = ?", (id_marca,))
        conn.commit()
        return cur.rowcount > 0
    finally:
        conn.close()


def create_modelo(nome_modelo: str, id_marca: int, lugares: int = 5, ano_fabricacao: int = None) -> int:
    conn = get_conn()
    cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO modelos_veiculos (nome_modelo, id_marca, lugares, ano_fabricacao)
            VALUES (?, ?, ?, ?)
        """, (nome_modelo, id_marca, lugares, ano_fabricacao))
        conn.commit()
        return cur.lastrowid
    except sqlite3.IntegrityError:
        # tentar recuperar existente (mesmo nome e marca)
        cur.execute("SELECT id_modelo FROM modelos_veiculos WHERE nome_modelo = ? AND id_marca = ?", (nome_modelo, id_marca))
        row = cur.fetchone()
        return row['id_modelo'] if row else None
    finally:
        conn.close()

def update_modelo(id_modelo: int, nome_modelo: str = None, id_marca: int = None,
                  lugares: int = None, ano_fabricacao: int = None) -> bool:
    allowed = {}
    if nome_modelo is not None:
        allowed['nome_modelo'] = nome_modelo.strip()
    if id_marca is not None:
        allowed['id_marca'] = int(id_marca)
    if lugares is not None:
        allowed['lugares'] = int(lugares)
    if ano_fabricacao is not None:
        allowed['ano_fabricacao'] = int(ano_fabricacao)

    if not allowed:
        return False

    set_parts = ", ".join([f"{k} = ?" for k in allowed.keys()])
    params = list(allowed.values()) + [id_modelo]

    conn = get_conn()
    cur = conn.cursor()
    try:
        cur.execute(f"UPDATE modelos_veiculos SET {set_parts} WHERE id_modelo = ?", params)
        conn.commit()
        return cur.rowcount > 0
    finally:
        conn.close()

def delete_modelo(id_modelo: int) -> bool:
    conn = get_conn()
    cur = conn.cursor()
    try:
        cur.execute("DELETE FROM modelos_veiculos WHERE id_modelo = ?", (id_modelo,))
        conn.commit()
        return cur.rowcount > 0
    finally:
        conn.close()


def create_cor(nome_cor: str) -> int:
    if not nome_cor or not nome_cor.strip():
        return None
    conn = get_conn()
    cur = conn.cursor()
    try:
        cur.execute("INSERT INTO cores_veiculos (nome_cor) VALUES (?)", (nome_cor.strip(),))
        conn.commit()
        return cur.lastrowid
    except sqlite3.IntegrityError:
        cur.execute("SELECT id_cor FROM cores_veiculos WHERE nome_cor = ?", (nome_cor.strip(),))
        row = cur.fetchone()
        return row['id_cor'] if row else None
    finally:
        conn.close()

def update_cor(id_cor: int, novo_nome: str) -> bool:
    if not str(id_cor).isdigit() or not novo_nome or not novo_nome.strip():
        return False
    conn = get_conn()
    cur = conn.cursor()
    try:
        cur.execute("UPDATE cores_veiculos SET nome_cor = ? WHERE id_cor = ?", (novo_nome.strip(), id_cor))
        conn.commit()
        return cur.rowcount > 0
    finally:
        conn.close()

def delete_cor(id_cor: int) -> bool:
    conn = get_conn()
    cur = conn.cursor()
    try:
        cur.execute("DELETE FROM cores_veiculos WHERE id_cor = ?", (id_cor,))
        conn.commit()
        return cur.rowcount > 0
    finally:
        conn.close()

def list_marcas_modelos_cores():
    conn = get_conn()
    cur = conn.cursor()

    print("\n--- Marcas ---")
    cur.execute("SELECT id_marca, nome_marca FROM marcas_veiculos ORDER BY id_marca")
    marcas = [dict(r) for r in cur.fetchall()]
    for m in marcas:
        print(f"[{m['id_marca']}] {m['nome_marca']}")

    print("\n--- Modelos (com marca) ---")
    cur.execute("""
        SELECT mo.id_modelo, mo.nome_modelo, mo.lugares, mo.ano_fabricacao, ma.id_marca, ma.nome_marca
        FROM modelos_veiculos mo
        LEFT JOIN marcas_veiculos ma ON mo.id_marca = ma.id_marca
        ORDER BY mo.id_modelo
    """)
    modelos = [dict(r) for r in cur.fetchall()]
    for mo in modelos:
        marca_info = f"{mo['nome_marca']}" if mo.get('nome_marca') else "Marca desconhecida"
        print(f"[{mo['id_modelo']}] {mo['nome_modelo']} | Marca: {marca_info} | Lugares: {mo.get('lugares')} | Ano: {mo.get('ano_fabricacao')}")

    print("\n--- Cores ---")
    cur.execute("SELECT id_cor, nome_cor FROM cores_veiculos ORDER BY id_cor")
    cores = [dict(r) for r in cur.fetchall()]
    for c in cores:
        print(f"[{c['id_cor']}] {c['nome_cor']}")

    conn.close()
    # opcional: retornar os dados para uso programático
    return {"marcas": marcas, "modelos": modelos, "cores": cores}


# Interface do MENU

In [6]:
# --- Interface de console (menu) ---
import sys
import time

def _print_motoristas_summary():
    rows = list_motoristas()
    if not rows:
        print("Nenhum motorista cadastrado.")
        return
    for r in rows:
        print(f"[{r['id_motorista']}] {r['nome_completo']} | CPF: {r.get('cpf')} | CNH: {r.get('cnh')} | Ativo: {bool(r['ativo'])}")

def _print_veiculos_summary():
    rows = list_veiculos()
    if not rows:
        print("Nenhum veículo cadastrado.")
        return
    for r in rows:
        print(f"[{r['id_veiculo']}] Placa: {r['placa']} | Motorista: [{r['id_motorista']}] {r.get('motorista')} | Modelo: {r.get('modelo')} | Cor: {r.get('cor')} | Ativo: {bool(r['ativo'])}")

def _input_int(prompt):
    v = input(prompt).strip()
    return int(v) if v.isdigit() else None


In [7]:
def _menu_help():
    print("""
Comandos disponíveis:
 1  - Listar motoristas
 2  - Ver detalhes motorista
 3  - Criar motorista
 4  - Atualizar motorista
 5  - Deletar motorista
 6  - Listar veículos
 7  - Criar veículo
 9  - Deletar veículo
10  - Listar marcas/modelos/cores
11  - Adicionar telefone motorista
12  - Adicionar endereço motorista

Gestão de catálogo:
13  - Criar marca
14  - Atualizar marca
15  - Deletar marca
16  - Criar modelo
17  - Atualizar modelo
18  - Deletar modelo
19  - Criar cor
20  - Atualizar cor
21  - Deletar cor

h   - Mostrar este menu
q   - Sair
""")

def run_console():
    _menu_help()
    while True:
        try:
            cmd = input("\nEscolha (h para ajuda): ").strip().lower()
            if cmd in ('q','0','sair'):
                print("Saindo...")
                break

            if cmd in ('h','help'):
                _menu_help()
                continue

            # ---------- Motoristas ----------
            if cmd == '1':
                _print_motoristas_summary()

            elif cmd == '2':
                mid = _input_int("ID do motorista: ")
                if not mid:
                    print("ID inválido.")
                    continue
                m = get_motorista(mid)
                if not m:
                    print("Motorista não encontrado.")
                    continue
                print("\n--- Detalhe Motorista ---")
                for k,v in m.items():
                    print(f"{k}: {v}")
                conn = get_conn()
                cur = conn.cursor()
                cur.execute("SELECT numero, tipo_telefone FROM telefones_motoristas WHERE id_motorista = ?", (mid,))
                phones = cur.fetchall()
                print("Telefones:")
                for p in phones:
                    print(" -", p['numero'], f"({p['tipo_telefone']})")
                cur.execute("SELECT rua, numero, complemento, bairro, cidade, estado, cep FROM enderecos_motoristas WHERE id_motorista = ?", (mid,))
                addrs = cur.fetchall()
                print("Endereços:")
                for a in addrs:
                    print(" -", a['rua'], a['numero'], a['bairro'], a['cidade'], a['estado'], a['cep'])
                conn.close()

            elif cmd == '3':
                print("Criar motorista (será solicitado pelo menos 1 telefone e 1 endereço).")
                nome = input("Nome completo: ").strip()
                if not nome:
                    print("Nome é obrigatório. Operação cancelada.")
                    continue
                cpf = input("CPF (opcional): ").strip() or None
                cnh = input("CNH (opcional): ").strip() or None
                data_nasc = input("Data Nasc YYYY-MM-DD (opcional): ").strip() or None
                email = input("Email (opcional): ").strip() or None
                ativo = input("Ativo? (s/n) [s]: ").strip().lower() or 's'
                ativo_val = 1 if ativo in ('s','sim','') else 0

                conn = get_conn()
                cur = conn.cursor()
                try:
                    conn.execute("BEGIN")
                    # inserir motorista
                    cur.execute("""
                        INSERT INTO motoristas (nome_completo, cpf, cnh, data_nasc, email, ativo)
                        VALUES (?, ?, ?, ?, ?, ?)
                    """, (nome, cpf, cnh, data_nasc, email, ativo_val))
                    mid = cur.lastrowid
                    print(f"Motorista criado com id: {mid}")

                    # telefones: pelo menos 1
                    telefones_adicionados = 0
                    while True:
                        numero = input("Telefone (formato livre) [obrigatório]: ").strip()
                        if numero:
                            tipo_tel = input("Tipo (celular/residencial/comercial) [celular]: ").strip() or 'celular'
                            cur.execute("INSERT INTO telefones_motoristas (id_motorista, numero, tipo_telefone) VALUES (?, ?, ?)",
                                        (mid, numero, tipo_tel))
                            telefones_adicionados += 1
                        else:
                            print("Telefone vazio. É necessário informar pelo menos um telefone.")
                            continue
                        more = input("Adicionar outro telefone? (s/n) [n]: ").strip().lower() or 'n'
                        if more not in ('s','sim'):
                            break
                    if telefones_adicionados == 0:
                        raise Exception("Pelo menos um telefone deve ser cadastrado.")

                    # endereços: pelo menos 1
                    enderecos_adicionados = 0
                    while True:
                        print("Informe um endereço (pelo menos 1):")
                        rua = input("  Rua: ").strip()
                        numero_end = input("  Número: ").strip()
                        complemento = input("  Complemento (opcional): ").strip() or None
                        bairro = input("  Bairro: ").strip()
                        cidade = input("  Cidade: ").strip()
                        estado = input("  Estado: ").strip()
                        cep = input("  CEP: ").strip()
                        if not rua or not cidade:
                            print("Rua e Cidade são obrigatórios para o endereço. Tente novamente.")
                            continue
                        cur.execute("""
                            INSERT INTO enderecos_motoristas (id_motorista, rua, numero, complemento, bairro, cidade, estado, cep)
                            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                        """, (mid, rua, numero_end, complemento, bairro, cidade, estado, cep))
                        enderecos_adicionados += 1
                        more = input("Adicionar outro endereço? (s/n) [n]: ").strip().lower() or 'n'
                        if more not in ('s','sim'):
                            break
                    if enderecos_adicionados == 0:
                        raise Exception("Pelo menos um endereço deve ser cadastrado.")

                    # tudo ok: commit
                    conn.commit()
                    print("Cadastro completo: motorista, telefones e endereços inseridos com sucesso.")
                except Exception as e:
                    conn.rollback()
                    print("Erro durante o cadastro. Operação revertida. Motivo:", e)
                finally:
                    conn.close()

            elif cmd == '4':
                mid = _input_int("ID do motorista a atualizar: ")
                if not mid:
                    print("ID inválido.")
                    continue
                m = get_motorista(mid)
                if not m:
                    print("Motorista não encontrado.")
                    continue
                print("Deixe em branco para manter o valor atual.")
                nome = input(f"Nome [{m['nome_completo']}]: ").strip() or m['nome_completo']
                cpf = input(f"CPF [{m.get('cpf')}]: ").strip() or m.get('cpf')
                cnh = input(f"CNH [{m.get('cnh')}]: ").strip() or m.get('cnh')
                data_nasc = input(f"Data Nasc [{m.get('data_nasc')}]: ").strip() or m.get('data_nasc')
                email = input(f"Email [{m.get('email')}]: ").strip() or m.get('email')
                ativo = input(f"Ativo (s/n) [{'s' if m['ativo'] else 'n'}]: ").strip().lower() or ('s' if m['ativo'] else 'n')
                ativo_val = 1 if ativo in ('s','sim') else 0
                try:
                    ok = update_motorista(mid, nome_completo=nome, cpf=cpf, cnh=cnh, data_nasc=data_nasc, email=email, ativo=ativo_val)
                    print("Atualizado." if ok else "Nenhuma alteração.")
                except Exception as e:
                    print("Erro ao atualizar:", e)

            elif cmd == '5':
                mid = _input_int("ID do motorista a deletar: ")
                if not mid:
                    print("ID inválido.")
                    continue
                try:
                    deleted = delete_motorista(mid)
                    print("Motorista deletado." if deleted else "Motorista não encontrado.")
                except Exception as e:
                    print("Não foi possível deletar:", e)

            # ---------- Veículos ----------
            elif cmd == '6':
                _print_veiculos_summary()

            elif cmd == '7':
                # Criar veículo: mostrar opções e permitir criar nova marca/modelo/cor
                print("Criar veículo (você pode escolher marca/modelo/cor existentes ou criar novos).")
                # mostrar marcas/modelos/cores
                dados = list_marcas_modelos_cores()  # já imprime e retorna dados

                # escolher/ criar marca
                escolha_marca = input("Escolha ID da marca existente ou digite 'nova' para criar nova marca: ").strip().lower()
                if escolha_marca == 'nova':
                    nome_marca = input("Nome da nova marca: ").strip()
                    if not nome_marca:
                        print("Marca inválida. Operação cancelada.")
                        continue
                    id_marca = create_marca(nome_marca)
                    print(f"Marca criada com id {id_marca}.")
                else:
                    id_marca = int(escolha_marca) if escolha_marca.isdigit() else None
                    if id_marca is None:
                        print("Marca inválida. Operação cancelada.")
                        continue

                # mostrar modelos da marca escolhida (filtrar)
                conn = get_conn()
                cur = conn.cursor()
                cur.execute("""
                    SELECT id_modelo, nome_modelo, ano_fabricacao FROM modelos_veiculos
                    WHERE id_marca = ?
                    ORDER BY id_modelo
                """, (id_marca,))
                modelos_da_marca = cur.fetchall()
                if modelos_da_marca:
                    print("\nModelos disponíveis para a marca escolhida (id - nome - ano):")
                    for mo in modelos_da_marca:
                        print(f"[{mo['id_modelo']}] {mo['nome_modelo']} - Ano: {mo['ano_fabricacao']}")
                else:
                    print("\nNenhum modelo cadastrado para essa marca.")

                escolha_modelo = input("Escolha ID do modelo existente ou digite 'novo' para criar novo modelo: ").strip().lower()
                if escolha_modelo == 'novo':
                    nome_modelo = input("Nome do novo modelo: ").strip()
                    ano_modelo = input("Ano de fabricação (opcional): ").strip()
                    ano_val = int(ano_modelo) if ano_modelo.isdigit() else None
                    lugares = input("Número de lugares (padrão 5): ").strip()
                    lugares_val = int(lugares) if lugares.isdigit() else 5
                    id_modelo = create_modelo(nome_modelo, id_marca, lugares_val, ano_val)
                    print(f"Modelo criado com id {id_modelo}.")
                else:
                    id_modelo = int(escolha_modelo) if escolha_modelo.isdigit() else None
                    if id_modelo is None:
                        print("Modelo inválido. Operação cancelada.")
                        conn.close()
                        continue

                # cores: listar e permitir nova
                cur.execute("SELECT id_cor, nome_cor FROM cores_veiculos ORDER BY id_cor")
                cores = cur.fetchall()
                print("\nCores disponíveis:")
                for c in cores:
                    print(f"[{c['id_cor']}] {c['nome_cor']}")
                escolha_cor = input("Escolha ID da cor existente ou digite 'nova' para criar nova cor: ").strip().lower()
                if escolha_cor == 'nova':
                    nome_cor = input("Nome da nova cor: ").strip()
                    if not nome_cor:
                        print("Cor inválida. Operação cancelada.")
                        conn.close()
                        continue
                    id_cor = create_cor(nome_cor)
                    print(f"Cor criada com id {id_cor}.")
                else:
                    id_cor = int(escolha_cor) if escolha_cor.isdigit() else None
                    if id_cor is None:
                        print("Cor inválida. Operação cancelada.")
                        conn.close()
                        continue

                # dados do veículo
                id_motorista = _input_int("ID do motorista (obrigatório): ")
                if not id_motorista:
                    print("ID motorista inválido.")
                    conn.close()
                    continue
                placa = input("Placa (obrigatório): ").strip()
                if not placa:
                    print("Placa obrigatória. Operação cancelada.")
                    conn.close()
                    continue
                ativo = input("Ativo? (s/n) [s]: ").strip().lower() or 's'
                ativo_val = 1 if ativo in ('s','sim','y','yes','') else 0

                try:
                    vid = create_veiculo(id_motorista, placa, id_modelo, id_cor, ativo_val)
                    print("Veículo criado com id:", vid)
                except Exception as e:
                    print("Erro ao criar veículo:", e)
                finally:
                    conn.close()

            elif cmd == '8':
                vid = _input_int("ID do veículo a atualizar: ")
                if not vid:
                    print("ID inválido.")
                    continue
                v = get_veiculo(vid)
                if not v:
                    print("Veículo não encontrado.")
                    continue
                print("Deixe em branco para manter o valor atual.")
                placa = input(f"Placa [{v['placa']}]: ").strip() or v['placa']
                id_motorista = input(f"ID Motorista [{v['id_motorista']}]: ").strip() or v['id_motorista']
                id_modelo = input(f"ID Modelo [{v.get('id_modelo')}]: ").strip() or v.get('id_modelo')
                id_cor = input(f"ID Cor [{v.get('id_cor')}]: ").strip() or v.get('id_cor')
                ativo = input(f"Ativo (s/n) [{'s' if v['ativo'] else 'n'}]: ").strip().lower() or ('s' if v['ativo'] else 'n')
                ativo_val = 1 if ativo in ('s','sim') else 0
                try:
                    ok = update_veiculo(vid, placa=placa, id_motorista=int(id_motorista), id_modelo=(int(id_modelo) if str(id_modelo).isdigit() else None), id_cor=(int(id_cor) if str(id_cor).isdigit() else None), ativo=ativo_val)
                    print("Atualizado." if ok else "Nenhuma alteração.")
                except Exception as e:
                    print("Erro ao atualizar veículo:", e)

            elif cmd == '9':
                vid = _input_int("ID do veículo a deletar: ")
                if not vid:
                    print("ID inválido.")
                    continue
                try:
                    deleted = delete_veiculo(vid)
                    print("Veículo deletado." if deleted else "Veículo não encontrado.")
                except Exception as e:
                    print("Erro ao deletar veículo:", e)

            # ---------- Catálogo: Marca / Modelo / Cor ----------
            elif cmd == '10':
                list_marcas_modelos_cores()

            elif cmd == '11':
                mid = _input_int("ID do motorista: ")
                if not mid:
                    print("ID inválido.")
                    continue
                numero = input("Número: ").strip()
                tipo = input("Tipo (celular/residencial/comercial) [celular]: ").strip() or 'celular'
                try:
                    conn = get_conn()
                    conn.execute("INSERT INTO telefones_motoristas (id_motorista, numero, tipo_telefone) VALUES (?, ?, ?)", (mid, numero, tipo))
                    conn.commit()
                    conn.close()
                    print("Telefone adicionado.")
                except Exception as e:
                    print("Erro ao adicionar telefone:", e)

            elif cmd == '12':
                mid = _input_int("ID do motorista: ")
                if not mid:
                    print("ID inválido.")
                    continue
                rua = input("Rua: ").strip()
                numero = input("Número: ").strip()
                complemento = input("Complemento (opcional): ").strip() or None
                bairro = input("Bairro: ").strip()
                cidade = input("Cidade: ").strip()
                estado = input("Estado: ").strip()
                cep = input("CEP: ").strip()
                try:
                    conn = get_conn()
                    conn.execute("""
                        INSERT INTO enderecos_motoristas (id_motorista, rua, numero, complemento, bairro, cidade, estado, cep)
                        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                    """, (mid, rua, numero, complemento, bairro, cidade, estado, cep))
                    conn.commit()
                    conn.close()
                    print("Endereço adicionado.")
                except Exception as e:
                    print("Erro ao adicionar endereço:", e)

            # ---------- Marca ----------
            elif cmd == '13':
                nome = input("Nome da nova marca: ").strip()
                if not nome:
                    print("Nome inválido.")
                    continue
                try:
                    mid = create_marca(nome)
                    print("Marca criada com id:", mid)
                except Exception as e:
                    print("Erro ao criar marca:", e)

            elif cmd == '14':
                id_marca = _input_int("ID da marca a atualizar: ")
                if not id_marca:
                    print("ID inválido.")
                    continue
                novo = input("Novo nome da marca: ").strip()
                if not novo:
                    print("Nome inválido.")
                    continue
                try:
                    ok = update_marca(id_marca, novo)
                    print("Marca atualizada." if ok else "Marca não encontrada ou sem alteração.")
                except Exception as e:
                    print("Erro ao atualizar marca:", e)

            elif cmd == '15':
                id_marca = _input_int("ID da marca a deletar: ")
                if not id_marca:
                    print("ID inválido.")
                    continue
                confirm = input("Confirma exclusão da marca? (s/n): ").strip().lower() or 'n'
                if confirm not in ('s','sim','y','yes'):
                    print("Operação cancelada.")
                    continue
                try:
                    deleted = delete_marca(id_marca)
                    print("Marca deletada." if deleted else "Marca não encontrada.")
                except Exception as e:
                    print("Não foi possível deletar marca:", e)

            # ---------- Modelo ----------
            elif cmd == '16':
                # criar modelo (associa a uma marca existente ou cria nova marca)
                dados = list_marcas_modelos_cores()
                escolha_marca = input("Escolha ID da marca existente para associar ou digite 'nova' para criar nova marca: ").strip().lower()
                if escolha_marca == 'nova':
                    nome_marca = input("Nome da nova marca: ").strip()
                    if not nome_marca:
                        print("Marca inválida. Operação cancelada.")
                        continue
                    id_marca = create_marca(nome_marca)
                    print(f"Marca criada com id {id_marca}.")
                else:
                    id_marca = int(escolha_marca) if escolha_marca.isdigit() else None
                    if id_marca is None:
                        print("Marca inválida. Operação cancelada.")
                        continue
                nome_modelo = input("Nome do modelo: ").strip()
                if not nome_modelo:
                    print("Nome do modelo inválido.")
                    continue
                ano = input("Ano de fabricação (opcional): ").strip()
                ano_val = int(ano) if ano.isdigit() else None
                lugares = input("Número de lugares (padrão 5): ").strip()
                lugares_val = int(lugares) if lugares.isdigit() else 5
                try:
                    mid = create_modelo(nome_modelo, id_marca, lugares_val, ano_val)
                    print("Modelo criado com id:", mid)
                except Exception as e:
                    print("Erro ao criar modelo:", e)

            elif cmd == '17':
                id_modelo = _input_int("ID do modelo a atualizar: ")
                if not id_modelo:
                    print("ID inválido.")
                    continue
                print("Deixe em branco para manter o valor atual.")
                nome_modelo = input("Novo nome do modelo: ").strip() or None
                id_marca = input("Novo ID da marca (opcional): ").strip() or None
                lugares = input("Novo número de lugares (opcional): ").strip() or None
                ano = input("Novo ano de fabricação (opcional): ").strip() or None
                try:
                    ok = update_modelo(
                        id_modelo,
                        nome_modelo=nome_modelo if nome_modelo is not None else None,
                        id_marca=int(id_marca) if id_marca and id_marca.isdigit() else None,
                        lugares=int(lugares) if lugares and lugares.isdigit() else None,
                        ano_fabricacao=int(ano) if ano and ano.isdigit() else None
                    )
                    print("Modelo atualizado." if ok else "Modelo não encontrado ou sem alteração.")
                except Exception as e:
                    print("Erro ao atualizar modelo:", e)

            elif cmd == '18':
                id_modelo = _input_int("ID do modelo a deletar: ")
                if not id_modelo:
                    print("ID inválido.")
                    continue
                confirm = input("Confirma exclusão do modelo? (s/n): ").strip().lower() or 'n'
                if confirm not in ('s','sim'):
                    print("Operação cancelada.")
                    continue
                try:
                    deleted = delete_modelo(id_modelo)
                    print("Modelo deletado." if deleted else "Modelo não encontrado.")
                except Exception as e:
                    print("Erro ao deletar modelo:", e)

            # ---------- Cor ----------
            elif cmd == '19':
                nome_cor = input("Nome da nova cor: ").strip()
                if not nome_cor:
                    print("Nome inválido.")
                    continue
                try:
                    cid = create_cor(nome_cor)
                    print("Cor criada com id:", cid)
                except Exception as e:
                    print("Erro ao criar cor:", e)

            elif cmd == '20':
                id_cor = _input_int("ID da cor a atualizar: ")
                if not id_cor:
                    print("ID inválido.")
                    continue
                novo_nome = input("Novo nome da cor: ").strip()
                if not novo_nome:
                    print("Nome inválido.")
                    continue
                try:
                    ok = update_cor(id_cor, novo_nome)
                    print("Cor atualizada." if ok else "Cor não encontrada ou sem alteração.")
                except Exception as e:
                    print("Erro ao atualizar cor:", e)

            elif cmd == '21':
                id_cor = _input_int("ID da cor a deletar: ")
                if not id_cor:
                    print("ID inválido.")
                    continue
                confirm = input("Confirma exclusão da cor? (s/n): ").strip().lower() or 'n'
                if confirm not in ('s','sim','y','yes'):
                    print("Operação cancelada.")
                    continue
                try:
                    deleted = delete_cor(id_cor)
                    print("Cor deletada." if deleted else "Cor não encontrada.")
                except Exception as e:
                    print("Erro ao deletar cor:", e)

            else:
                print("Comando não reconhecido. Digite 'h' para ajuda.")

        except KeyboardInterrupt:
            print("\nInterrompido pelo usuário. Saindo.")
            break
        except Exception as e:
            print("Erro inesperado:", e)


# Execução do programa



In [ ]:
run_console()